In [84]:
import torch
import re
from collections import defaultdict

In [85]:
text = open("train.txt", "r", encoding="utf-8").read()

unique_chars = sorted(list(set(text)))
vocab_size = len(unique_chars)


print(''.join(unique_chars))
print(f"\nvocab_Size:{vocab_size}")

# unique_chars.extend(['4','6','7','9','0'])
# Adding remaining numbers 

def word_tokenize(text):
    """Each word is a token, but also each special character, spaces arent ocnsidered."""
    # Split on spaces but keep punctuation as separate tokens
    tokens = re.findall(r"\w+|[^\w\s]", text)
    return tokens
word_tokens = word_tokenize(text)


print(f"\nFirst 50 WORD tokens:\n {word_tokens[:50]}")




 !"&'(),-./12358:;?ABCDEFGHIJKLMNOPQRSTUVWYabcdefghijklmnopqrstuvwxyzé

vocab_Size:71

First 50 WORD tokens:
 ['A', 'SCANDAL', 'IN', 'BOHEMIA', 'Arthur', 'Conan', 'Doyle', 'Table', 'of', 'contents', 'Chapter', '1', 'Chapter', '2', 'Chapter', '3', 'CHAPTER', 'I', 'To', 'Sherlock', 'Holmes', 'she', 'is', 'always', 'the', 'woman', '.', 'I', 'have', 'seldom', 'heard', 'him', 'mention', 'her', 'under', 'any', 'other', 'name', '.', 'In', 'his', 'eyes', 'she', 'eclipses', 'and', 'predominates', 'the', 'whole', 'of', 'her']


In [86]:
def split_words_with_sep(text):
    for m in re.finditer(r'(\S+)(\s*)', text):
        word, sep = m.groups()
        if not word:
            continue
        marker = '<para>' if sep.count('\n') >= 2 else '<end>'
        yield word, marker

def get_vocab(text):
    vocab = defaultdict(int)
    for word, marker in split_words_with_sep(text):
        vocab[' '.join(list(word)) + ' ' + marker] += 1
    return vocab

def get_stats(vocab):
    """Count frequency of every adjacent pair and stores in dict pairs"""
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_mostfrequent(best_pair, vocab):
    """Merge the most frequent pair across all words, variable- pair is the most freq adjacent pair"""
    new_vocab = {}
    to_replace = ' '.join(best_pair)
    replacement = ''.join(best_pair)
    for word in vocab:
        new_word = word.replace(to_replace, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab


def train_bpe(text, n_merges):
    vocab = get_vocab(text)
    merges = []

    for i in range(n_merges):
        pairs = get_stats(vocab)
        if not pairs:
            break
        # merge the most frequent pair
        best_pair = max(pairs, key=pairs.get)
        vocab = merge_mostfrequent(best_pair, vocab)
        merges.append(best_pair)

    return vocab, merges

print("\n\n")
bpe_vocab, merges = train_bpe(text, n_merges=300)

# Build final token list from BPE vocab
bpe_tokens = set()
for word in bpe_vocab:
    for token in word.split():
        bpe_tokens.add(token)

bpe_tokens = sorted(list(bpe_tokens))
print(f"BPE vocab size: {len(bpe_tokens)}")
print(f"First 20 tokens: {bpe_tokens[:20]}")

stoi = {token: i for i, token in enumerate(bpe_tokens)}
itos = {i: token for i, token in enumerate(bpe_tokens)}

bpe_vocab_size=len(bpe_tokens)
#string ot integer and integer to string

def encode(text):
    token_ids = []
    for word, marker in split_words_with_sep(text):
        symbols = list(word) + [marker]
        for pair in merges:
            i = 0
            while i < len(symbols) - 1:
                if symbols[i] == pair[0] and symbols[i+1] == pair[1]:
                    symbols = symbols[:i] + [''.join(pair)] + symbols[i+2:]
                else:
                    i += 1
        for token in symbols:
            if token in stoi:
                token_ids.append(stoi[token])
    return token_ids

encoded = encode(text)

def decode(token_ids):
    tokens = [itos[i] for i in token_ids]
    text = ''.join(tokens)
    text = text.replace('<para>', '\n\n').replace('<end>', ' ')
    return text.strip()

n = int(0.9 * len(encoded))
train = encoded[:n]
test = encoded[n:]


train = torch.tensor(train, dtype=torch.long)
test = torch.tensor(test, dtype=torch.long)

print(f"Train tokens: {len(train)}, Test tokens: {len(test)}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")






BPE vocab size: 1342
First 20 tokens: ['!', '!"<end>', '!<end>', '"', '"<end>', '"<para>', '"And<end>', '"But<end>', '"I<end>', '"It<end>', '"N', '"T', '"The<end>', '"W', '"You<end>', '&', "'", "'<end>", "'s<end>", '(']
Train tokens: 20053, Test tokens: 2229
Device: CPU


In [87]:
torch.manual_seed(10)
block_size = 64  # context length
batch_size = 16  # no of sampling chunks in one forward pass

def get_batch(data):
    #random start positions
    ix = torch.randint(len(data) - block_size, (batch_size,))
    
    # x is the input context, y is the target (shifted by 1)
    x = torch.stack([data[i : i+block_size] for i in ix])
    y = torch.stack([data[i+1 : i+block_size+1] for i in ix])
    
    return x, y

xb, yb = get_batch(train)
print(xb.shape)
print(yb.shape,"\n\n")

print("INPUTS",xb)
print("OUTPUTS:",yb)

    

torch.Size([16, 64])
torch.Size([16, 64]) 


INPUTS tensor([[ 649,  101, 1175,  ...,   28,   90,  672],
        [  92,  824,  825,  ..., 1298,  825,  261],
        [ 456,  648,   27,  ..., 1078,  427,  528],
        ...,
        [ 783,  871,   38,  ...,  421,  520,  538],
        [ 421,  101,  707,  ...,  847,  726,  315],
        [ 187, 1300, 1141,  ...,  183,  538,  649]])
OUTPUTS: tensor([[ 101, 1175,  577,  ...,   90,  672,  515],
        [ 824,  825,  667,  ...,  825,  261, 1174],
        [ 648,   27,    8,  ...,  427,  528, 1298],
        ...,
        [ 871,   38,  577,  ...,  520,  538, 1269],
        [ 101,  707, 1192,  ...,  726,  315,  277],
        [1300, 1141,  635,  ...,  538,  649,  693]])


In [ ]:
torch.manual_seed(10)
import torch.nn as nn  

import torch.nn.functional as F
n_embed=48
dropout=0.2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

In [89]:
# Self Attention Head:
# Every token spits out two vectors, one is the key vector and the other is the query vector. The key vector is used to represent the token itself, 
# the query vector is used to represent the token's relationship with prev toks.
# Query . Key(eachtoken) gives a score that represents how much attention should be paid to each token when predicting the next token. 
#  score is softmaxxed to get PDF over all tokens.
# This is  used to compute a weighted sum of the value vectors of all tokens, which gives the final output for that token.

# weight matrix becomes that once our head is done.

In [ ]:
#Attention HEAD.
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.head_size = head_size
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)

        # Scale by sqrt(head_size) — not sqrt(C) — so the scale matches the
        # dimension being dot-producted (head_size), not the full embedding dim.
        weight = q @ k.transpose(-2, -1) * (self.head_size ** -0.5)  # (B, T, T)
        weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weight = F.softmax(weight, dim=-1)                             # (B, T, T)
        weight = self.dropout(weight)

        v = self.value(x)  # (B, T, head_size)
        return weight @ v  # (B, T, head_size)

In [ ]:
# Multi-Head Self-Attention
# Run num_heads independent attention heads in parallel, each in a
# head_size = n_embed // num_heads subspace.
# Concatenate their outputs (restoring n_embed width) and project back.
# Each head can specialise on different positional / semantic relationships.

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj  = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Each head: (B, T, head_size); cat along last dim → (B, T, n_embed)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))  # (B, T, n_embed)
    



# Repetitive blocks of self attention and feed forward layers:

# Feed-Forward: per-token MLP, as per the "Attention Is All You Need" paper.
# Expands to 4x n_embed, applies a nonlinearity, then projects back down to
# n_embed so it can be added back into the residual stream.
class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding dimension, n_head: the number of heads
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))#communication
        x = x + self.ffwd(self.ln2(x))#computation
        return x

In [ ]:
class BigramLM(nn.Module):

    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table    = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        # 4 heads, each of size n_embed // 4
        self.blocks=nn.Sequential(
            Block(n_embed, n_head=4),
            Block(n_embed, n_head=4)
        )  # 2 blocks
        self.ln_f      = nn.LayerNorm(n_embed)
        self.lm_head   = nn.Linear(n_embed, vocab_size)
        self.token_embedding_table.weight = self.lm_head.weight  # weight tying

    def forward(self, index, targets=None):
        tok_emb = self.token_embedding_table(index)                                                # (B, T, n_embed)
        pos_emb = self.position_embedding_table(torch.arange(index.size(1), device=index.device)) # (T, n_embed)
        x       = tok_emb + pos_emb # (B, T, n_embed=C)
        
        x       = self.blocks(x)    # (B, T, n_embed)
        x       = self.ln_f(x)      # (B, T, n_embed)
        logits  = self.lm_head(x)   # (B, T, vocab_size)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits  = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss    = F.cross_entropy(logits, targets, label_smoothing=0.1)
        return logits, loss

    def generate(self, index, max_new_tokens):
        for _ in range(max_new_tokens):
            index_cond  = index[:, -block_size:]
            logits, _   = self.forward(index_cond)
            logits      = logits[:, -1, :]
            probs       = F.softmax(logits, dim=-1)
            next_token  = torch.multinomial(probs, num_samples=1)
            index       = torch.cat([index, next_token], dim=1)
        return index

model = BigramLM(bpe_vocab_size, n_embed).to(DEVICE)
xb, yb = xb.to(DEVICE), yb.to(DEVICE)
logits, loss = model(xb, yb)
print(logits.shape)
print(loss)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
batch_size = 32
best_loss = 10
eval_iters = 50  # number of batches to average over when estimating loss

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split, data in [('train', train), ('val', test)]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(data)
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

for i in range(25000):
    xb, yb = get_batch(train)
    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
    logits, loss = model(xb, yb)    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if i % 500 == 0:
        losses = estimate_loss()
        print(f"step {i}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        if losses['val'] < best_loss:
            best_loss = losses['val']
            torch.save(model.state_dict(), 'bigram_weights.pth')
            
losses = estimate_loss()
print(f"Training done. Best loss: {best_loss:.4f} | final train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")


In [94]:
# To capture more info about previous tokens, we can use multiple ways.
#  We can aggregate the mean of all previous tokens instead of just using the previous token for predicting the next token,

# xagg=torch.zeros((B,T,C))
# for b in range(B):
#     for t in range(T):
#         xprev=x[b,:t+1,:]
#         xagg[b,t]=torch.mean(xprev,dim=0)

# xagg[b,t] stores all previous info till t (excluding t). This is a simple way to give the model more context, but it can be improved.

# print(x[0],"\n\n",xagg[0])



#This can be done using matrix multiplicaiton or softmax in a much more efficient matter instead of adding.

In [95]:
# torch.manual_seed(10)
# B,T,C=4,8,32
# x=torch.randn(B,T,C)
# x.shape

# tril=torch.tril(torch.ones(T,T))
# weight=torch.zeros((T,T))
# weight=weight.masked_fill(tril==0,float('-inf'))  # future values are nerfed because we only concentrate till present token.
# weight=F.softmax(weight,dim=-1)
# out=weight@x

# out.shape 

In [96]:
model.load_state_dict(torch.load('bigram_weights.pth', map_location=DEVICE))
model.eval()

context = torch.tensor([encode("She")], dtype=torch.long, device=DEVICE)
generated = model.generate(context, max_new_tokens=200)
print(decode(generated[0].tolist()))

She probe less which must alwere alk a soled his broad.

"I cer!" Thusper."

"I am aughp. I was ont at at in the tentiminether, whenay."

"Ston anght?" His ot is ofrienan her I shotion to ad," a retion of it. "Tpaigently for him. You may was, the me. And it was shietly, st ight me lare of the other ender iector, illent to mon."

"And come then?"

"I tare leiting-he will gal three gal it was alpriciming from his stion Orp of eindoub
